# 04 - Analise de Erros

Diagnostico detalhado dos erros do modelo para identificar fraquezas.

**Nota:** Requer `python scripts/retrain.py` executado previamente.

In [ ]:
import sys

sys.path.append('..')

import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from src.features.feature_engineering import FeatureEngineer
from src.models.evaluation import ModelEvaluator

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)

## 1. Carregar Modelo e Dados

In [ ]:
models_dir = Path('../data/models')

with open(models_dir / 'features' / 'feature_names.txt') as f:
    saved_features = [line.strip() for line in f.readlines() if line.strip()]

model = joblib.load(models_dir / 'checkpoints' / 'best_model.pkl')
print("Modelo carregado: best_model.pkl")
print(f"Features esperadas: {len(saved_features)}")

df = pd.read_parquet('../data/processed/processed_data.parquet')
fe = FeatureEngineer()
df_features = fe.create_all_features(df)

available_features = [f for f in saved_features if f in df_features.columns]
missing = [f for f in saved_features if f not in df_features.columns]
if missing:
    print(f"AVISO: {len(missing)} features em falta: {missing}")

print(f"Features usadas: {len(available_features)}")

## 2. Preparar Conjunto de Teste

In [ ]:
df_sorted = df_features.sort_values('timestamp').reset_index(drop=True)
test_start = int(0.85 * len(df_sorted))
df_test = df_sorted.iloc[test_start:].copy()

X_test = df_test[available_features].values
y_test = df_test['consumption_mw'].values

y_pred = model.predict(X_test)

df_test['y_pred'] = y_pred
df_test['error'] = y_test - y_pred
df_test['abs_error'] = np.abs(df_test['error'])
df_test['pct_error'] = (df_test['error'] / y_test) * 100
df_test['abs_pct_error'] = np.abs(df_test['pct_error'])

df_test['hour'] = df_test['timestamp'].dt.hour
df_test['day_of_week'] = df_test['timestamp'].dt.dayofweek
df_test['month_val'] = df_test['timestamp'].dt.month
df_test['season'] = df_test['month_val'].map(
    lambda m: 'Inverno' if m in [12,1,2] else 'Primavera' if m in [3,4,5] else 'Verao' if m in [6,7,8] else 'Outono'
)

evaluator = ModelEvaluator()
metrics = evaluator.calculate_metrics(y_test, y_pred)
print("\nMetricas globais:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")
print(f"  Amostras: {len(y_test):,}")

## 3. Erro por Regiao

In [ ]:
regional = df_test.groupby('region').agg(
    mae=('abs_error', 'mean'),
    rmse=('error', lambda x: np.sqrt(np.mean(x**2))),
    mape=('abs_pct_error', 'mean'),
    count=('error', 'count')
).sort_values('mape')

print("=" * 70)
print("ERRO POR REGIAO")
print("=" * 70)
print(regional.to_string())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
regional['mae'].plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('MAE por Regiao (MW)')
axes[0].set_ylabel('MAE (MW)')

regional['mape'].plot(kind='bar', ax=axes[1], color='coral', edgecolor='black')
axes[1].set_title('MAPE por Regiao (%)')
axes[1].set_ylabel('MAPE (%)')

regional['rmse'].plot(kind='bar', ax=axes[2], color='green', edgecolor='black')
axes[2].set_title('RMSE por Regiao (MW)')
axes[2].set_ylabel('RMSE (MW)')

for ax in axes:
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 4. Erro por Hora

In [ ]:
hourly = df_test.groupby('hour').agg(
    mae=('abs_error', 'mean'),
    mape=('abs_pct_error', 'mean')
).sort_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar(hourly.index, hourly['mae'], color='steelblue', edgecolor='black')
axes[0].set_xlabel('Hora')
axes[0].set_ylabel('MAE (MW)')
axes[0].set_title('MAE por Hora do Dia')
axes[0].set_xticks(range(0, 24))

axes[1].bar(hourly.index, hourly['mape'], color='coral', edgecolor='black')
axes[1].set_xlabel('Hora')
axes[1].set_ylabel('MAPE (%)')
axes[1].set_title('MAPE por Hora do Dia')
axes[1].set_xticks(range(0, 24))

plt.tight_layout()
plt.show()

print(f"Pior hora (MAPE): {hourly['mape'].idxmax()}h ({hourly['mape'].max():.2f}%)")
print(f"Melhor hora (MAPE): {hourly['mape'].idxmin()}h ({hourly['mape'].min():.2f}%)")

## 5. Erro por Estacao

In [ ]:
seasonal = df_test.groupby('season').agg(
    mae=('abs_error', 'mean'),
    mape=('abs_pct_error', 'mean'),
    count=('error', 'count')
)

print("\nErro por Estacao:")
print(seasonal.to_string())

fig, ax = plt.subplots(figsize=(8, 5))
seasonal['mape'].plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_ylabel('MAPE (%)')
ax.set_title('MAPE por Estacao')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

## 6. Analise de Residuos

In [ ]:
residuals = df_test['error'].values

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].plot(df_test['timestamp'].values[:2000], residuals[:2000], alpha=0.5, linewidth=1)
axes[0,0].axhline(0, color='red', linestyle='--')
axes[0,0].set_title('Residuos ao Longo do Tempo')
axes[0,0].set_ylabel('Residuo (MW)')

axes[0,1].hist(residuals, bins=80, edgecolor='black', alpha=0.7, density=True)
axes[0,1].axvline(0, color='red', linestyle='--')
axes[0,1].set_title('Distribuicao dos Residuos')
axes[0,1].set_xlabel('Residuo (MW)')

axes[1,0].scatter(y_pred[:5000], residuals[:5000], alpha=0.3, s=10)
axes[1,0].axhline(0, color='red', linestyle='--')
axes[1,0].set_xlabel('Valor Previsto (MW)')
axes[1,0].set_ylabel('Residuo (MW)')
axes[1,0].set_title('Residuos vs Previsao')

stats.probplot(residuals, dist="norm", plot=axes[1,1])
axes[1,1].set_title('Q-Q Plot')

plt.tight_layout()
plt.show()

print("\nEstatisticas dos Residuos:")
print(f"  Media (bias):  {residuals.mean():.2f} MW")
print(f"  Desvio-padrao: {residuals.std():.2f} MW")
print(f"  Skewness:      {stats.skew(residuals):.4f}")
print(f"  Kurtosis:      {stats.kurtosis(residuals):.4f}")

## 7. Resumo

In [ ]:
print("=" * 70)
print("RESUMO DA ANALISE DE ERROS")
print("=" * 70)
print(f"""
Metricas Globais:
  MAE:  {metrics['mae']:.2f} MW
  RMSE: {metrics['rmse']:.2f} MW
  MAPE: {metrics['mape']:.2f}%
  R2:   {metrics['r2']:.4f}

Padroes Identificados:
  - Pior regiao (MAPE): {regional['mape'].idxmax()} ({regional['mape'].max():.2f}%)
  - Melhor regiao: {regional['mape'].idxmin()} ({regional['mape'].min():.2f}%)
  - Pior hora: {hourly['mape'].idxmax()}h ({hourly['mape'].max():.2f}%)
  - Melhor hora: {hourly['mape'].idxmin()}h ({hourly['mape'].min():.2f}%)
  - Bias medio: {residuals.mean():.2f} MW (proximo de zero = bom)
""")